# E se eu tivesse investido R\\$ 200 todo mês?

**Python Finance Lab · Aula 3** · *Finanças na prática, notebook por notebook.*

---

## Parte 1 · A pergunta

Nas aulas 1 e 2, a história começava com R\\$ 2.000 aplicados de uma vez, no primeiro dia. Para muita gente, principalmente para quem está começando, não é assim que funciona: o dinheiro para investir aparece aos poucos, a cada mês, conforme sobra do salário, da mesada ou da bolsa de estágio.

Nesta aula, vamos simular esse caminho: R\\$ 200 por mês, durante 60 meses, de outubro de 2021 a setembro de 2026, num total de R\\$ 12.000 aportados. A cada mês, os R\\$ 200 são divididos igualmente entre as 10 ações da carteira, R\\$ 20 em cada uma. Para comparar, aplicamos os mesmos R\\$ 200 por mês no Ibovespa e no CDI.

As perguntas são duas: no fim, quanto esses R\\$ 12.000 viraram? E o caminho foi diferente do de quem investiu tudo de uma vez?

Do lado do Python, a novidade é o **laço `for`**: a ferramenta para repetir uma tarefa, como comprar cotas, uma vez para cada mês.

*Conteúdo exclusivamente educacional. Não constitui recomendação de investimento.*

## Parte 2 · Conceitos

### Bloco 2 · O vocabulário da aula

**Aporte.** É cada depósito feito num investimento. Aqui, R\\$ 200 por mês, sempre no último pregão do mês: o primeiro em outubro de 2021 e o último em 30 de setembro de 2026.

**Cota.** Quem aplica num fundo não compra um valor em reais: compra **cotas**, pedacinhos do fundo, pelo preço do dia. O patrimônio é o número de cotas vezes o preço atual da cota. Vamos usar essa ideia nos três investimentos:

- **Nas ações**, a cota é a própria ação: R\\$ 20 compram uma fração de ação pelo preço do mês.
- **No Ibovespa**, imaginamos um fundo cuja cota vale exatamente os pontos do índice, em reais. Com R\\$ 200, compra-se uma fração minúscula dessa cota.
- **No CDI**, criamos uma cota que começa valendo cerca de 1 e cresce a cada dia útil com a taxa do CDI. É assim que funcionam, na prática, os fundos e títulos atrelados ao CDI.

**Preço médio.** Quem investe todo mês o mesmo valor em reais leva mais cotas quando o preço está baixo e menos quando está alto. Por isso, o preço médio pago por cota fica abaixo da média simples dos preços do período (ou igual, se o preço nunca mudar). Em inglês, essa estratégia de aportes regulares se chama *dollar-cost averaging*.

**Total aportado × patrimônio.** O total aportado é a soma dos depósitos: 60 × R\\$ 200 = R\\$ 12.000. O patrimônio é quanto esse dinheiro vale. A diferença entre os dois é o ganho, ou a perda.

**Por que o percentual engana.** Na aula 1, os R\\$ 2.000 ficaram cinco anos investidos. Aqui, o primeiro aporte fica quase cinco anos, e o último, nenhum dia. Em média, cada real fica investido cerca de dois anos e meio. Por isso, o ganho em % do total aportado não pode ser comparado diretamente com o retorno da aula 1.

## Parte 3 · Fórmulas

### Bloco 3 · As contas da aula

A cada mês $m$, com aporte $A$ e preço da cota $P_m$:

$$q_m = \frac{A}{P_m} \qquad\qquad Q_m = Q_{m-1} + q_m \qquad\qquad W_m = Q_m \times P_m$$

- $q_m$: cotas compradas no mês;
- $Q_m$: cotas acumuladas até o mês;
- $W_m$: patrimônio no mês;
- total aportado até o mês: $m \times A$.

Na carteira, cada ação recebe $A/N$ = R\\$ 20 por mês e é acompanhada separadamente. O patrimônio da carteira é a soma das 10 ações.

*Exemplo à mão:* aportes de R\\$ 100 por mês, com o preço da cota em 10, 8 e 12.

| Mês | Preço | Cotas compradas | Cotas acumuladas | Patrimônio | Total aportado |
|---|---|---|---|---|---|
| 1 | 10 | 10,00 | 10,00 | R\\$ 100,00 | R\\$ 100 |
| 2 | 8 | 12,50 | 22,50 | R\\$ 180,00 | R\\$ 200 |
| 3 | 12 | 8,33 | 30,83 | R\\$ 370,00 | R\\$ 300 |

No mês 2, o preço caiu e o patrimônio ficou abaixo do total aportado. Mas o aporte desse mês comprou mais cotas, e, quando o preço subiu, elas valorizaram. O preço médio foi de R\\$ 300 ÷ 30,83 cotas = R\\$ 9,73, abaixo da média simples dos três preços (R\\$ 10).

## Parte 4 · Implementação

### Bloco 4 · Importar as bibliotecas

As mesmas bibliotecas das aulas anteriores.

> **O que esperar:** nada. Se aparecer `ModuleNotFoundError`, instale com `%pip install yfinance pandas matplotlib requests` e reinicie o kernel (menu Kernel → Restart Kernel).

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import requests

### Bloco 5 · As premissas

As mesmas ações e datas das aulas anteriores. No lugar do capital inicial, entra o aporte mensal. O `print` confere quanto cada ação recebe por mês.

> **O que esperar:** `Por ação, a cada mês: R$ 20.0`.

In [ ]:
ACOES = [
    "ITUB4.SA", "VALE3.SA", "PETR4.SA", "WEGE3.SA", "ABEV3.SA",
    "EGIE3.SA", "SUZB3.SA", "RADL3.SA", "VIVT3.SA", "TOTS3.SA",
]
INDICE = "^BVSP"
DATA_INICIO = "2021-10-01"
DATA_FIM = "2026-09-30"
APORTE_MENSAL = 200        # R$ 200 todo mês
print("Por ação, a cada mês: R$", APORTE_MENSAL / len(ACOES))

### Bloco 6 · Os dados da aula 1

O mesmo bloco de dados da aula 2: preços ajustados das ações e do Ibovespa, do Yahoo Finance, e CDI diário, do Banco Central.

> **O que esperar:** nada. Os dados ficam em `precos`, `ibov` e `cdi_diario`.

In [ ]:
# Ações e Ibovespa (Yahoo Finance), como na aula 1
dados = yf.download(ACOES, start=DATA_INICIO, auto_adjust=True, progress=False)
precos = dados["Close"].loc[DATA_INICIO:DATA_FIM].ffill()
precos.columns = precos.columns.str.replace(".SA", "")

dados_ibov = yf.download(
    INDICE, start=DATA_INICIO, auto_adjust=True, progress=False, multi_level_index=False
)
ibov = dados_ibov["Close"].loc[DATA_INICIO:DATA_FIM]

# CDI (Banco Central), como na aula 1
inicio_bc = pd.to_datetime(DATA_INICIO).strftime("%d/%m/%Y")
fim_bc = pd.to_datetime(DATA_FIM).strftime("%d/%m/%Y")
url = (
    "https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados"
    f"?formato=json&dataInicial={inicio_bc}&dataFinal={fim_bc}"
)
resposta = requests.get(url, timeout=60)
resposta.raise_for_status()
cdi_bruto = pd.DataFrame(resposta.json())
cdi_bruto["data"] = pd.to_datetime(cdi_bruto["data"], format="%d/%m/%Y")
cdi_bruto["valor"] = cdi_bruto["valor"].astype(float)
cdi_diario = cdi_bruto.set_index("data")["valor"]

### Bloco 7 · A cota do CDI

Para tratar o CDI do mesmo jeito que as ações e o Ibovespa, ele precisa de um "preço". É a cota do CDI: o produto acumulado dos fatores diários, como na aula 1, só que sem multiplicar pelo capital. Ela começa perto de 1 e cresce um pouco a cada dia útil.

Comprar "cotas do CDI" com R\\$ 200 e acompanhar o valor delas é exatamente o que acontece com quem aplica num investimento que rende o CDI.

> **O que esperar:** os cinco primeiros valores da cota, todos ligeiramente acima de 1 e crescendo dia após dia.

In [ ]:
cota_cdi = (1 + cdi_diario / 100).cumprod()
cota_cdi.head()

### Bloco 8 · Os preços de fim de mês

Os aportes são mensais, então precisamos do preço de cada mês, e não de cada dia. O `resample` reagrupa uma série temporal em períodos maiores:

- `"ME"`, de *month end* (fim do mês), agrupa por mês;
- `.last()` pega o último valor de cada mês, que é o preço do último pregão.

Aplicamos o mesmo às ações, ao Ibovespa e à cota do CDI.

Repare num detalhe da tabela: as datas aparecem como o último dia do calendário, por exemplo 31/10/2021, mesmo que o último pregão tenha sido em outro dia (29/10/2021, uma sexta-feira). A etiqueta indica o mês; o preço é o do último pregão dele.

Nas versões do pandas anteriores à 2.2, o código do fim do mês era `"M"`. Hoje, é `"ME"`.

> **O que esperar:** `Meses: 60` e as cinco primeiras linhas da tabela mensal das ações, de outubro de 2021 a fevereiro de 2022.

In [ ]:
precos_mes = precos.resample("ME").last()
ibov_mes = ibov.resample("ME").last()
cdi_mes = cota_cdi.resample("ME").last()

print("Meses:", len(precos_mes))
precos_mes.head()

### Bloco 9 · O primeiro laço

Antes de simular os aportes, vamos entender o `for` com um exemplo simples. O laço percorre uma lista, item por item. A cada volta, a variável `mes` recebe o próximo item, e o bloco recuado logo abaixo é executado com esse valor. Três itens, três voltas, três linhas impressas.

O recuo funciona como nas funções da aula 2: tudo o que está recuado pertence ao laço.

> **O que esperar:** três linhas: `Aporte de R$ 200 em outubro`, `Aporte de R$ 200 em novembro` e `Aporte de R$ 200 em dezembro`.

In [ ]:
for mes in ["outubro", "novembro", "dezembro"]:
    print("Aporte de R$", APORTE_MENSAL, "em", mes)

### Bloco 10 · Passo a passo: três meses de Ibovespa

Agora o laço de verdade, só nos três primeiros meses do Ibovespa e imprimindo cada etapa. É a mesma conta da tabela feita à mão no bloco 3.

- `cotas = 0` começa a contagem antes do laço: no início, não temos nenhuma cota.
- `ibov_mes.index[:3]` são as três primeiras datas. O `[:3]` significa "do começo até a posição 3, sem incluí-la".
- A cada volta, buscamos o preço do mês com `.loc[data]`, calculamos as cotas compradas, somamos ao total e mostramos o patrimônio.

A linha `cotas = cotas + compradas` é o coração do laço: o novo valor de `cotas` é o valor antigo mais as cotas compradas no mês. É assim que o laço "lembra" o que aconteceu nas voltas anteriores.

Nas f-strings, `:.2f` mostra duas casas decimais e `:.6f` mostra seis, porque as frações de cota do Ibovespa são muito pequenas.

> **O que esperar:** três linhas, uma por mês. No primeiro mês, o patrimônio é exatamente R\\$ 200. Nos seguintes, ele já é diferente do total aportado (R\\$ 400 e R\\$ 600), porque as cotas compradas antes mudaram de valor.

In [ ]:
cotas = 0
for data in ibov_mes.index[:3]:
    preco = ibov_mes.loc[data]
    compradas = APORTE_MENSAL / preco
    cotas = cotas + compradas
    print(
        data.date(),
        f"| cota: {preco:.2f}",
        f"| compradas: {compradas:.6f}",
        f"| acumuladas: {cotas:.6f}",
        f"| patrimônio: R$ {cotas * preco:.2f}",
    )

### Bloco 11 · Os 60 meses do Ibovespa

O mesmo laço, agora em todos os meses e sem os `print`s. Para guardar o patrimônio de cada mês, usamos uma lista que começa vazia, `patrimonio = []`, e que, a cada volta, recebe mais um valor com `.append()`, o método que acrescenta um item ao final da lista.

Depois do laço, transformamos a lista numa Series, usando as datas dos meses como índice. O resultado é o patrimônio mês a mês.

> **O que esperar:** o patrimônio dos últimos cinco meses de quem aportou R\\$ 200 por mês no Ibovespa.

In [ ]:
cotas = 0
patrimonio = []
for data in ibov_mes.index:
    preco = ibov_mes.loc[data]
    cotas = cotas + APORTE_MENSAL / preco
    patrimonio.append(cotas * preco)

ibov_aportes = pd.Series(patrimonio, index=ibov_mes.index)
ibov_aportes.tail()

### Bloco 12 · A função que simula aportes

Precisamos repetir essa simulação para o CDI e para cada uma das 10 ações. Na aula 2, aprendemos o que fazer com uma conta que se repete: transformá-la em função.

A função `simular_aportes` recebe os preços mensais de um investimento e o valor do aporte, e devolve o patrimônio mês a mês. Dentro dela está exatamente o laço do bloco anterior, com `aporte` no lugar de `APORTE_MENSAL`.

Como toda função nova, ela é testada com exemplos de resposta conhecida:

- **Preço constante de R\\$ 10, com aportes de R\\$ 100:** sem variação de preço, o patrimônio é igual ao total aportado: 100, 200, 300.
- **Preço que dobra de R\\$ 1 para R\\$ 2:** os primeiros R\\$ 100 viram R\\$ 200, que, somados ao novo aporte de R\\$ 100, dão R\\$ 300.

O `.tolist()` transforma a série numa lista, só para a exibição ficar curta.

> **O que esperar:** `[100.0, 200.0, 300.0]` e `[100.0, 300.0]`.

In [ ]:
def simular_aportes(precos_mensais, aporte):
    """Investe o mesmo valor no fim de cada mês e devolve o patrimônio mês a mês."""
    cotas = 0
    patrimonio = []
    for data in precos_mensais.index:
        preco = precos_mensais.loc[data]
        cotas = cotas + aporte / preco
        patrimonio.append(cotas * preco)
    return pd.Series(patrimonio, index=precos_mensais.index)


print(simular_aportes(pd.Series([10.0, 10.0, 10.0]), 100).tolist())   # esperado: [100, 200, 300]
print(simular_aportes(pd.Series([1.0, 2.0]), 100).tolist())           # esperado: [100, 300]

### Bloco 13 · Ibovespa, CDI e as 10 ações

Com a função pronta, o Ibovespa e o CDI cabem numa linha cada.

Para a carteira, usamos um segundo laço, agora sobre as ações. Dividir R\\$ 200 entre 10 ações todo mês é o mesmo que manter 10 mini-carteiras, cada uma recebendo R\\$ 20 por mês. Simulamos cada mini-carteira com a função e vamos somando os resultados:

- `precos_mes.columns` é a lista com os nomes das 10 ações, e o laço percorre essa lista.
- `precos_mes[acao]` é a coluna de preços da ação da vez.
- `carteira_aportes` começa em zero. Na primeira volta, zero mais uma série dá a própria série; nas seguintes, o pandas soma série com série, mês a mês.

Usamos `len(ACOES)` em vez de escrever 10, para que o código continue certo se a carteira mudar de tamanho.

> **O que esperar:** o patrimônio da carteira nos últimos cinco meses.

In [ ]:
ibov_aportes = simular_aportes(ibov_mes, APORTE_MENSAL)
cdi_aportes = simular_aportes(cdi_mes, APORTE_MENSAL)

aporte_por_acao = APORTE_MENSAL / len(ACOES)
carteira_aportes = 0
for acao in precos_mes.columns:
    carteira_aportes = carteira_aportes + simular_aportes(precos_mes[acao], aporte_por_acao)

carteira_aportes.tail()

### Bloco 14 · O total aportado e a tabela da evolução

Para comparar, precisamos do total aportado em cada mês. `pd.Series(APORTE_MENSAL, index=precos_mes.index)` cria uma série com o valor 200 em todos os meses, e o `cumsum()`, a **soma acumulada**, a transforma em 200, 400, 600… até 12.000. Ele é o irmão do `cumprod` da aula 1: em vez de multiplicar, soma.

Depois, juntamos tudo numa tabela com `pd.concat`, como na aula 1.

> **O que esperar:** uma tabela com 60 linhas, uma por mês, e quatro colunas. Na primeira linha, todas valem 200.

In [ ]:
total_aportado = pd.Series(APORTE_MENSAL, index=precos_mes.index).cumsum()

evolucao = pd.concat(
    [carteira_aportes, ibov_aportes, cdi_aportes, total_aportado], axis=1, join="inner"
)
evolucao.columns = ["Carteira", "Ibovespa", "CDI", "Total aportado"]
evolucao

### Bloco 15 · Patrimônio × total aportado

O gráfico mostra o patrimônio das três alternativas e, em cinza, o total aportado: uma rampa que sobe R\\$ 200 por mês. Quando uma curva está acima da rampa, aquele investimento está ganhando; abaixo, perdendo. Cores e estilos de linha são os mesmos das aulas anteriores.

> **O que esperar:** quatro linhas subindo juntas no começo e se separando com o tempo.

*Nota de ambiente:* com pandas 2.2 e matplotlib 3.11 ou mais novo, gráficos com datas mensais podem exibir um aviso amarelo (`MatplotlibDeprecationWarning`) que vem de dentro do próprio pandas. Ele não altera o gráfico e não aparece com o pandas 3.

In [ ]:
ax = evolucao.plot(
    figsize=(10, 5),
    style=["-", "--", ":", "-"],
    color=["#2a78d6", "#eb6834", "#1baf7a", "#898781"],
    linewidth=2,
    title=f"R$ {APORTE_MENSAL} por mês: patrimônio × total aportado",
)
ax.set_ylabel("R$")
ax.set_xlabel("")
ax.grid(alpha=0.3)
plt.show()

**O que o gráfico mostra.** **[Preencher: qual alternativa terminou na frente e em que períodos a carteira ficou abaixo da rampa.]**

### Bloco 16 · A tabela final

A resposta em números: quanto foi aportado, quanto virou e qual foi o ganho, em reais e em % do total aportado.

- `final = evolucao.iloc[-1]` pega a última linha, de setembro de 2026.
- `final[["Carteira", "Ibovespa", "CDI"]]` seleciona os três patrimônios. Os colchetes duplos indicam uma lista de colunas.
- Duas colunas são calculadas a partir de outras colunas da própria tabela: o ganho é o patrimônio menos o aportado, e o ganho em % é o ganho dividido pelo aportado.

> **O que esperar:** uma tabela com três linhas e quatro colunas. O total aportado é R\\$ 12.000 nas três.

In [ ]:
final = evolucao.iloc[-1]

resumo = pd.DataFrame(index=["Carteira", "Ibovespa", "CDI"])
resumo["Total aportado (R$)"] = final["Total aportado"]
resumo["Patrimônio final (R$)"] = final[["Carteira", "Ibovespa", "CDI"]]
resumo["Ganho (R$)"] = resumo["Patrimônio final (R$)"] - resumo["Total aportado (R$)"]
resumo["Ganho sobre o aportado (%)"] = resumo["Ganho (R$)"] / resumo["Total aportado (R$)"] * 100
resumo.round(2)

**A resposta em números.** **[Preencher com os números.]**

### Bloco 17 · Ganho ou perda acumulada

O gráfico do bloco 15 tem uma limitação: como o total aportado cresce o tempo todo, as curvas sobem mesmo quando o investimento está perdendo. Para enxergar o resultado de verdade, subtraímos o total aportado do patrimônio. O que sobra é o ganho (acima de zero) ou a perda (abaixo de zero) acumulada até cada mês.

O `.sub(..., axis=0)` subtrai a coluna do total aportado de cada uma das três colunas, linha a linha. O `axis=0` indica que o alinhamento é pelas linhas, ou seja, pelas datas.

Por fim, contamos os meses em que a carteira ficou no vermelho, com o truque da aula 2: verdadeiro vale 1, então a soma conta quantos verdadeiros existem.

> **O que esperar:** três linhas a partir do zero. A do CDI sobe sempre; as outras alternam ganhos e perdas. Repare que, no fim do período, as oscilações em reais ficam maiores: há mais dinheiro investido, então a mesma variação percentual move mais reais.

In [ ]:
lucro_ou_prejuizo = evolucao[["Carteira", "Ibovespa", "CDI"]].sub(evolucao["Total aportado"], axis=0)

ax = lucro_ou_prejuizo.plot(
    figsize=(10, 4),
    style=["-", "--", ":"],
    color=["#2a78d6", "#eb6834", "#1baf7a"],
    linewidth=2,
    title="Ganho ou perda acumulada (patrimônio − total aportado)",
)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("R$")
ax.set_xlabel("")
ax.grid(alpha=0.3)
plt.show()

meses_no_vermelho = (lucro_ou_prejuizo["Carteira"] < 0).sum()
print("Meses em que a carteira valia menos do que o total aportado:", meses_no_vermelho, "de", len(evolucao))

**Os meses no vermelho.** **[Preencher: em quantos meses a carteira ficou no vermelho.]**

## Parte 5 · Validação

### Bloco 18 · As contas fecham?

Três conferências rápidas:

1. **60 aportes**, um por mês, de outubro de 2021 a setembro de 2026.
2. **Total aportado de R\\$ 12.000.**
3. **CDI nunca abaixo do total aportado.** Como o CDI nunca cai, o patrimônio aplicado nele não pode valer menos do que foi depositado. O `.all()` responde se a condição vale para todos os meses.

No terceiro teste, o patrimônio do CDI é arredondado para centavos com `.round(2)` antes da comparação. Como vimos na aula 2, o computador guarda números decimais com pequenas imprecisões: no primeiro mês, `200 / preco * preco` pode dar `199.99999999999997` em vez de 200, e o teste falharia sem que houvesse erro nenhum. Comparar em centavos, como faz qualquer extrato bancário, resolve.

> **O que esperar:** `60`, `12000` e `True`.

In [ ]:
print("Número de aportes:", len(evolucao))
print("Total aportado: R$", evolucao["Total aportado"].iloc[-1])
print("CDI sempre acima do aportado?", (evolucao["CDI"].round(2) >= evolucao["Total aportado"]).all())

### Bloco 19 · A mesma conta sem laço

O laço deixa cada passo visível, e é o melhor jeito de aprender a lógica. Mas o pandas consegue fazer a mesma conta sem laço, com a tabela inteira de uma vez:

- `aporte_por_acao / precos_mes` calcula as cotas compradas de cada ação em todos os meses;
- `.cumsum()` acumula as cotas mês a mês;
- multiplicando pelos preços e somando as 10 colunas com `sum(axis=1)`, temos o patrimônio da carteira.

Se as duas contas estiverem certas, a diferença entre elas é zero, ou um número minúsculo, por arredondamento.

> **O que esperar:** `Maior diferença entre as duas contas: R$ 0.0`.

In [ ]:
cotas_acumuladas = (aporte_por_acao / precos_mes).cumsum()
carteira_sem_laco = (cotas_acumuladas * precos_mes).sum(axis=1)

diferenca = (carteira_sem_laco - carteira_aportes).abs().max()
print("Maior diferença entre as duas contas: R$", round(diferenca, 6))

## Parte 6 · Interpretação

### Bloco 20 · O que os números dizem

**[Texto a completar com os resultados reais.]**

Aportando R\\$ 200 por mês durante 60 meses, foram investidos R\\$ 12.000. Na carteira, eles viraram **R\\$ [x]**, um ganho de **R\\$ [x]**, ou **[x]%** do total aportado. No Ibovespa, viraram **R\\$ [x]**; no CDI, **R\\$ [x]**.

Perguntas que orientam a interpretação:

1. **A ordem entre as alternativas foi a mesma da aula 1?** Se mudou, a explicação provavelmente está no fim do período: quem aporta todo mês tem mais dinheiro investido nos últimos anos, então o que acontece no fim pesa mais no resultado.
2. **Por quantos meses a carteira ficou no vermelho?** Como seria atravessar esses meses vendo o saldo abaixo do que você depositou?
3. **Por que o ganho em % é diferente do retorno da aula 1?** Porque cada real ficou investido, em média, cerca de dois anos e meio, e não cinco.
4. **O que os aportes regulares fazem com a sorte?** Cada mês comprou a um preço diferente: quem comprou no fundo de uma queda se deu bem, e quem comprou num pico, nem tanto. Aportar todo mês dilui essa sorte, para o bem e para o mal.

A medida mais adequada para comparar investimentos com aportes ao longo do tempo é a **taxa interna de retorno (TIR)**, que leva em conta por quanto tempo cada real ficou aplicado. Ela fica para uma aula futura.

## Parte 7 · Limitações

### Bloco 21 · O que esta análise não mostra

- **Frações de ação.** R\\$ 20 por ação por mês é menos do que custa uma única ação da maioria delas. Na prática, quem aporta R\\$ 200 compraria, a cada mês, uma ou duas ações inteiras no mercado fracionário, alternando entre elas. A simulação usa frações para isolar o efeito dos aportes; a aula 4 mostra um jeito prático de decidir em qual ação aportar.
- **Comparação bruta.** Não entram custos de corretagem, taxas da bolsa nem impostos.
- **Data do aporte.** Os aportes acontecem no último pregão de cada mês. Outra data muda um pouco o resultado (exercício 2).
- **Ibovespa sem custos.** Na prática, o caminho seria um fundo de índice, que cobra taxa.
- **Os cuidados das aulas anteriores continuam valendo:** o passado não garante o futuro, as ações foram escolhidas com o conhecimento de hoje e os dados do Yahoo Finance precisam de conferência.

## Parte 8 · Exercícios

### Bloco 22 · Para praticar

1. **Outro valor de aporte.** Mude `APORTE_MENSAL` para 100 e depois para 500. O patrimônio muda, mas o ganho em % não. Por quê?
2. **Aporte no início do mês.** Troque `"ME"` por `"MS"` (*month start*) e `.last()` por `.first()` no bloco 8. O resultado muda muito? Repare que a última avaliação passa a ser feita no começo de setembro de 2026.
3. **Preço médio.** Para uma ação, divida o total investido nela (60 × R\\$ 20) pelo total de cotas compradas e compare com a média simples dos preços mensais, `precos_mes["ITUB4"].mean()`. Qual é menor?
4. **Capital inicial mais aportes.** Crie uma nova versão da função com um parâmetro `inicial` e simule R\\$ 2.000 iniciais mais R\\$ 200 por mês. Dica: antes do laço, em vez de começar com zero cotas, compre as cotas do capital inicial pelo preço do primeiro mês.
5. **O pior mês.** Descubra em que mês a carteira teve a maior perda em relação ao total aportado, com `lucro_ou_prejuizo["Carteira"].idxmin()`.

## Resumo

### Bloco 23 · O que aprendemos

**Em finanças:**

- Aportar todo mês é comprar cotas pelo preço de cada mês; o patrimônio é o total de cotas vezes o preço atual.
- Aportes regulares compram mais cotas quando o preço está baixo, e o preço médio fica abaixo da média dos preços.
- O resultado se lê comparando o patrimônio com o total aportado, mês a mês.
- O ganho em % sobre o total aportado não se compara diretamente ao retorno de quem investiu tudo de uma vez.

**Em Python:**

- O laço `for` repete um bloco de código para cada item de uma lista.
- Uma variável atualizada dentro do laço (`cotas = cotas + ...`) guarda a memória das voltas anteriores.
- `.append()` acrescenta itens a uma lista; uma lista pode virar uma Series.
- `resample("ME").last()` transforma uma série diária em mensal; `cumsum` acumula somas.
- Um laço dentro de uma função a torna reutilizável, e um segundo laço pode chamar essa função várias vezes.
- A mesma conta feita de dois jeitos diferentes é uma ótima validação.

## Gancho para a aula 4

### Bloco 24 · Próxima aula

Repare numa coisa: a cada mês, colocamos exatamente R\\$ 20 em cada ação, independentemente de como ela estava. Com o tempo, as ações que mais subiram passaram a ocupar uma fatia maior da carteira, e as que caíram, uma fatia menor. Isso é bom ou ruim?

E se, em vez de dividir igualmente, usássemos o aporte do mês para reforçar as ações que ficaram para trás? Ou se, de tempos em tempos, vendêssemos um pouco das que subiram para voltar aos 10% em cada uma? Na próxima aula: rebalancear vale a pena? E o Python ganha o `if`, a forma de o código tomar decisões.

## Glossário da aula

| Termo | Significado |
|---|---|
| Aporte | Cada depósito feito num investimento |
| Cota | Fração de um investimento; patrimônio = cotas × preço da cota |
| Preço médio | Total investido dividido pelo total de cotas compradas |
| Total aportado | Soma de todos os depósitos |
| Patrimônio | Quanto o investimento vale num determinado dia |
| TIR | Taxa interna de retorno; considera quando cada real entrou |
| Laço `for` | Estrutura que repete um bloco de código para cada item de uma lista |
| `append` | Método que acrescenta um item ao final de uma lista |
| `resample` | Método que reagrupa uma série temporal em períodos maiores |
| `cumsum` | Soma acumulada |

---

*Python Finance Lab · Conteúdo exclusivamente educacional. Não constitui recomendação de investimento.*